In [2]:
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

# --- Load the CSV ---
df = pd.read_csv(
    "../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_all_p_value.csv"
)

# --- Clean up p-values (remove brackets from string representation) ---
pval_cols = ["test_bfs_p_val", "test_bfs_simple_p_val", "test_opt_mov_p_val"]
for col in pval_cols:
    df[col] = df[col].astype(str).str.replace("[", "").str.replace("]", "").astype(float)

# --- Global FDR correction across ALL tests (Option B) ---
all_pvals = df[pval_cols].values.flatten()
rejected, pvals_corrected, _, _ = multipletests(
    all_pvals,
    method="fdr_bh",
    alpha=0.05
)

# --- Create New DataFrame with Interleaved Columns ---
# We want: [metric, group, p1, sig1, p2, sig2, p3, sig3]
new_df = pd.DataFrame()
new_df['comparison_metric'] = df['comparison_metric']
new_df['patient_group'] = df['patient_group']

# Reconstruct the p-values into a 2D array (rows, columns)
# Shape is (number_of_rows, 3)
pvals_matrix = pvals_corrected.reshape(df.shape[0], len(pval_cols))
rejected_matrix = rejected.reshape(df.shape[0], len(pval_cols))

for i, col_name in enumerate(pval_cols):
    # Add the corrected p-value
    new_df[col_name] = pvals_matrix[:, i]
    # Add the corresponding significance label immediately after
    new_df[f"{col_name}_sig"] = np.where(rejected_matrix[:, i], "Rejected", "Accepted")

# --- Display ---
display(new_df)

# --- Save corrected table ---
new_df.to_csv(
    "../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_all_p_value_fdr.csv",
    index=False
)


FileNotFoundError: [Errno 2] No such file or directory: '../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_all_p_value.csv'

In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

# --- Define the 4 file paths here ---
file_paths = [
    "../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_easy_v_easy_p_value.csv",
    "../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_easy_v_hard_p_value.csv",
    "../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_hard_v_hard_p_value.csv",
]

pval_cols = ["test_bfs_p_val", "test_bfs_simple_p_val", "test_opt_mov_p_val"]
all_dfs = []

# --- Load and Preprocess ---
for i, path in enumerate(file_paths):
    try:
        df = pd.read_csv(path)
        # Add a column to identify the source table
        df['table_source'] = f"Table_{i+1}"
        
        # Clean up p-values (remove brackets and convert to float)
        for col in pval_cols:
            df[col] = df[col].astype(str).str.replace("[", "").str.replace("]", "").astype(float)
        
        all_dfs.append(df)
        print(f"Successfully loaded {path}")
    except Exception as e:
        print(f"Error loading {path}: {e}")

# --- Combine Tables ---
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"Combined shape: {combined_df.shape}")

    # --- Global FDR correction across ALL tests in all 4 tables ---
    # We flatten all p-values from the 3 columns across all rows into one 1D array
    all_pvals = combined_df[pval_cols].values.flatten()
    
    # Perform Benjamini-Hochberg FDR correction
    rejected, pvals_corrected, _, _ = multipletests(
        all_pvals,
        method="fdr_bh",
        alpha=0.05
    )

    # --- Create New Interleaved DataFrame ---
    # We want: [comparison_metric, patient_group, table_source, p1, sig1, p2, sig2, p3, sig3]
    new_df = pd.DataFrame()
    
    # Keep the identifier columns at the start
    base_cols = ["comparison_metric", "patient_group", "table_source"]
    for col in base_cols:
        new_df[col] = combined_df[col]

    # Reconstruct the p-values and rejection status into 2D matrices (rows, columns)
    pvals_matrix = pvals_corrected.reshape(combined_df.shape[0], len(pval_cols))
    rejected_matrix = rejected.reshape(combined_df.shape[0], len(pval_cols))

    for i, col_name in enumerate(pval_cols):
        # Add the corrected p-value
        new_df[col_name] = pvals_matrix[:, i]
        # Add the corresponding significance label immediately after
        new_df[f"{col_name}_sig"] = np.where(rejected_matrix[:, i], "Rejected", "Accepted")

    # --- Display ---
    display(new_df)

    # --- Save corrected table ---
    output_path = "../../../../tower_of_london_plots/tables/p_values/combined_tables_fdr_analysis.csv"
    new_df.to_csv(output_path, index=False)
    print(f"Saved corrected table to: {output_path}")
else:
    print("No tables were loaded. Please check your file paths.")


Error loading ../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_easy_v_easy_p_value.csv: [Errno 2] No such file or directory: '../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_easy_v_easy_p_value.csv'
Error loading ../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_easy_v_hard_p_value.csv: [Errno 2] No such file or directory: '../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_easy_v_hard_p_value.csv'
Error loading ../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_hard_v_hard_p_value.csv: [Errno 2] No such file or directory: '../../../../tower_of_london_plots/tables/p_values/table_val_bfs_opt_sim_22_2_120_top1_mtfnone_hard_v_hard_p_value.csv'
No tables were loaded. Please check your file paths.
